# Instruction tuning.

Models that have had `sft` applied are called `Instruct` models. I will show how that works be instruction tuning gpt2.

**This is a little more advance than prompting**

### I Will do what I did in the notebook titled `gpt2-base-model.ipynb` just in case it was skipped. I will show the reason models are need to have SFT

SFT is the same as instruction tuning. Instruction tuning is not the same as fine-tuning. Fine-tuning is when you update ALL of the weights on a pre-trained model. This should NEVER be done.

In [ ]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

prompts = (
    "who is the president of the US?",
    "how are you doing today?",
    "can you help me solve this",
    "please summarize the themes of 1984?",
    "how many films did Kubrick Make",
)

inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.pad_token_id,
)

for i, prompt in enumerate(prompts):
    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[i][prompt_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    print(f"==== Prompt {i} ====")
    print(prompt)
    print(f"==== Base model reply {i} ====")
    print(response)
    print()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


==== Prompt 0 ====
who is the president of the US?
==== Base model reply 0 ====
 the president of the United States today? what is the president's full name? who is the president's first name and last name? what is the president's full name? what is the president's first and last name? what is the president's

==== Prompt 1 ====
how are you doing today?
==== Base model reply 1 ====
 do you feel about your studies?
How do you feel about your studies?
How do you feel about your studies?
How do you feel about your studies?

Maybe you are feeling good about your studies. Let me know if you need anything else!

But

==== Prompt 2 ====
can you help me solve this
==== Base model reply 2 ====
 please help me solve this problem? How to solve for x in the equation 2x - 12 = 2x - 26?
Okay, let me try to figure out how to solve this equation. The equation given is 

==== Prompt 3 ====
please summarize the themes of 1984?
==== Base model reply 3 ====
 

The main themes of 1984 are surveillance, con

## Why SFT is important.

The about output is just predicting the next sequence of token based on the tokens is was given. Not very useful without post-training.

In [ ]:
import gc

del tokenizer, model, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

### Why this output is important.

Without SFT, the model is pretty much just auto complete. As more data is added and other things (I will get to) are added, the model does become better. Models need instruction-fine tuning to be able, to be simple, to mimic chat data.



# SFT Qwen

I will perform SFT on qwen using dataset name `databricks/databricks-dolly-15k` from hugging face hub. This is going to make it so the LLM has better responses.

In [ ]:
from huggingface_hub import login
from datasets import load_dataset

login()

ds = load_dataset("vietgpt/databricks_dolly15k_en")['train']
ds = ds.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = ds["train"], ds["test"]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM


tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="qwen-instruct-chat",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    max_length=256,
    learning_rate=5e-5,
    lr_scheduler_type="linear",
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    bf16=True,
    report_to="none",
)

model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

## Train The model

This should be done a a GPU that is not a consumer grade gpu.

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.360408,2.259825,2.288070,110818.000000,0.535959
200,2.268365,2.241216,2.262379,214990.000000,0.537321
300,2.292668,2.223709,2.242067,326575.000000,0.541226
400,2.252797,2.213521,2.253357,436039.000000,0.542537
500,2.193422,2.204769,2.149347,544324.000000,0.545454
600,2.312555,2.190154,2.253689,656773.000000,0.545657
700,2.173001,2.173373,2.142737,768622.000000,0.548262
800,2.243632,2.163869,2.149656,880120.000000,0.549935
900,2.113417,2.153328,2.142873,992544.000000,0.552967
1000,2.191457,2.147844,2.153690,1103675.000000,0.553769


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1783, training_loss=2.200366230941661, metrics={'train_runtime': 13555.7535, 'train_samples_per_second': 1.052, 'train_steps_per_second': 0.132, 'total_flos': 9382591085936640.0, 'train_loss': 2.200366230941661, 'epoch': 1.0})

### Save The Models

In [ ]:
output_dir = "qwen-instruct-chat"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen-instruct-chat/tokenizer_config.json',
 'qwen-instruct-chat/chat_template.jinja',
 'qwen-instruct-chat/tokenizer.json')

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()